# Paco's Binder — notebook console

A Python client over `underwritting_api`, doing the same things
[`underwritting_web`](../../underwritting_web) does from the browser — pick a product,
answer questions, evaluate a quote, browse/build the catalog — but from a notebook instead
of a page. Handy for scripting a batch of quotes, poking at the API interactively, or just
not wanting to open a browser.

**Before running this:** have `underwritting_api` running somewhere reachable (locally,
that's `uvicorn app.main:app --port 8000` from that repo with its `.env` loaded), and
install this folder's deps:

```bash
pip install -r notebooks/requirements.txt
```

Set `UW_API_URL` if the API isn't on the default `http://localhost:8000`.


In [ ]:
import os
from getpass import getpass

import pandas as pd
import requests

# same default underwritting_web/src/lib/api.js falls back to when VITE_API_URL isn't set
API_URL = os.environ.get("UW_API_URL", "http://localhost:8000")
print(f"talking to {API_URL}")


## API client

Same shape as `underwritting_web/src/lib/api.js` — one small function per endpoint, all
going through one request helper. Every error response on this API always has a string
`detail` (see `app/main.py`'s `RequestValidationError` handler), so `ApiError.detail` is
always something printable, no matter which layer raised it.


In [ ]:
class ApiError(Exception):
    # .detail is always a plain string, not a dict/list - same guarantee the frontend relies on
    def __init__(self, status: int, detail: str) -> None:
        super().__init__(f"{status}: {detail}")
        self.status = status
        self.detail = detail


def _request(
    method: str,
    path: str,
    *,
    json_body: dict | None = None,
    admin_key: str | None = None,
    quote_token: str | None = None,
) -> dict | list | None:
    # everything below funnels through here, same as api.js's request() on the frontend
    headers: dict[str, str] = {}
    if admin_key:
        headers["X-Admin-Key"] = admin_key
    if quote_token:
        headers["X-Quote-Token"] = quote_token

    resp = requests.request(method, f"{API_URL}{path}", json=json_body, headers=headers)
    data = resp.json() if resp.content else None

    if not resp.ok:
        detail = (data or {}).get("detail", resp.reason)
        raise ApiError(resp.status_code, detail)
    return data


In [ ]:
# -- public, applicant-facing - no key needed for any of these --

def get_products() -> list[dict]:
    return _request("GET", "/products")


def get_product_questions(product_code: str) -> list[dict]:
    return _request("GET", f"/products/{product_code}/questions")


def create_quote(product_code: str) -> dict:
    return _request("POST", "/quotes", json_body={"product_code": product_code})


def submit_answers(quote_id: int, quote_token: str, answers: dict[str, str]) -> dict:
    # answers here is just {question_code: answer_text} - friendlier to type out by hand
    # than the API's list-of-objects shape, converted right before the call
    payload = [{"question_code": code, "answer_text": text} for code, text in answers.items()]
    return _request(
        "POST",
        f"/quotes/{quote_id}/answers",
        json_body={"answers": payload},
        quote_token=quote_token,
    )


def evaluate_quote(quote_id: int, quote_token: str, strategy: str) -> dict:
    # strategy is "full" or "short_circuit"
    return _request(
        "POST",
        f"/quotes/{quote_id}/evaluate",
        json_body={"strategy": strategy},
        quote_token=quote_token,
    )


In [ ]:
# -- admin, behind X-Admin-Key --

def get_product_rules(product_code: str, admin_key: str) -> list[dict]:
    return _request("GET", f"/products/{product_code}/rules", admin_key=admin_key)


def create_product(admin_key: str, code: str, name: str, description: str | None = None) -> dict:
    body = {"code": code, "name": name, "description": description}
    return _request("POST", "/products", json_body=body, admin_key=admin_key)


def add_product_question(
    admin_key: str,
    product_code: str,
    question_code: str,
    text: str,
    answer_type: str,
    sequence: int,
    enum_options: list[str] | None = None,
    is_mandatory: bool = True,
    expected_answer: str | None = None,
) -> dict:
    body = {
        "question_code": question_code,
        "text": text,
        "answer_type": answer_type,
        "enum_options": enum_options,
        "sequence": sequence,
        "is_mandatory": is_mandatory,
        "expected_answer": expected_answer,
    }
    return _request("POST", f"/products/{product_code}/questions", json_body=body, admin_key=admin_key)


def add_product_rule(
    admin_key: str,
    product_code: str,
    name: str,
    outcome: str,
    conditions: list[dict],
    priority: int = 1,
    stop_evaluation: bool = False,
) -> dict:
    # each condition is {"question_code": ..., "operator": ..., "value": ...}
    body = {
        "name": name,
        "priority": priority,
        "outcome": outcome,
        "stop_evaluation": stop_evaluation,
        "conditions": conditions,
    }
    return _request("POST", f"/products/{product_code}/rules", json_body=body, admin_key=admin_key)


## Quote console

Same flow as the web app's Quote console tab: browse products, pick one, answer its
questions, submit, then evaluate under both models.


In [ ]:
products = get_products()
pd.DataFrame(products)


In [ ]:
# pick a product code from the table above and see its questions
product_code = "LIFE_SIMPLE"

questions = get_product_questions(product_code)
pd.DataFrame(questions)


In [ ]:
quote = create_quote(product_code)
quote_id, quote_token = quote["quote_id"], quote["access_token"]
print(f"quote_id={quote_id}")

# fill in an answer per question_code from the table above, then re-run this cell
answers = {
    "Q_SMOKER": "true",
    "Q_CANCER": "false",
    "Q_BMI": "34",
    "Q_HOSP": "false",
    "Q_SPORTS": "false",
}

submit_answers(quote_id, quote_token, answers)


In [ ]:
# run both evaluation models and print the trigger - same "which answer decided it" info
# the web app's session panel shows
for strategy in ("full", "short_circuit"):
    result = evaluate_quote(quote_id, quote_token, strategy)
    trigger = result["trigger"]

    print(f"--- {strategy} ---")
    print(f"outcome: {result['outcome']}")
    if trigger:
        stopped = " (stopped early - Model B didn't need to look further)" if trigger["stopped_early"] else ""
        print(f"decided by: {trigger['rule_name']} -> {trigger['question_codes']}{stopped}")
    else:
        print("decided by: nothing matched, this is the accept default")
    print()


### A quick look at the error handling

Same behavior the frontend leans on: a wrong `quote_token` and a nonexistent `quote_id`
come back as the exact same 404, so there's no way to tell them apart from the outside.


In [ ]:
try:
    evaluate_quote(quote_id, "not-the-real-token", "full")
except ApiError as exc:
    print(f"{exc.status}: {exc.detail}")


## Admin

Needs the real `ADMIN_API_KEY` from `underwritting_api/.env` — pasted in below via
`getpass` so it never ends up saved in this notebook's cell output.


In [ ]:
admin_key = getpass("Paste your ADMIN_API_KEY: ")


### Browse catalog — a product's real rules

`GET /products/{code}/rules` is admin-only (same reasoning as never exposing
`expected_answer` publicly — a rule's conditions are the exact thresholds that decide an
outcome).


In [ ]:
rules = get_product_rules(product_code, admin_key)
pd.json_normalize(rules)


### Build new — create a product, question, and rule

⚠️ **This makes real, permanent writes to the live catalog** — same as **Build new** in the
web app. There's no delete endpoint. Edit the values below, then uncomment and run when
you're actually ready to create something.


In [ ]:
# new_product = create_product(admin_key, code="BOAT_BASIC", name="Basic Boat Insurance")
# new_product

# new_question = add_product_question(
#     admin_key,
#     "BOAT_BASIC",
#     question_code="Q_BOAT_LENGTH",
#     text="Boat length (ft)?",
#     answer_type="number",
#     sequence=1,
# )
# new_question

# new_rule = add_product_rule(
#     admin_key,
#     "BOAT_BASIC",
#     name="Boat too long",
#     outcome="decline",
#     conditions=[{"question_code": "Q_BOAT_LENGTH", "operator": ">", "value": "40"}],
# )
# new_rule
